In [1]:
import pandas as pd
import os

print("Starting Player Fatigue Analysis...")

# 1. Load Data
# We use low_memory=False because IPL.csv is huge and has mixed data types
ipl = pd.read_csv('../data/raw/IPL.csv', low_memory=False)

# 2. Convert date to actual datetime objects for time-series math
ipl['date'] = pd.to_datetime(ipl['date'])

# 3. Extract the "Playing XIs" per match
# A player played if they batted, bowled, or were the non-striker
batters = ipl[['match_id', 'date', 'batter']].rename(columns={'batter': 'player'})
non_strikers = ipl[['match_id', 'date', 'non_striker']].rename(columns={'non_striker': 'player'})
bowlers = ipl[['match_id', 'date', 'bowler']].rename(columns={'bowler': 'player'})

# Combine them all and drop duplicates to get a clean list of who played in which match
match_players = pd.concat([batters, non_strikers, bowlers]).dropna().drop_duplicates()

# Sort chronologically for the rolling window to work correctly
match_players = match_players.sort_values(by=['player', 'date']).reset_index(drop=True)

# 4. Calculate Rolling 7-Day Match Count (The Fatigue Index)
# Set the date as the index so pandas can calculate rolling time windows
match_players.set_index('date', inplace=True)

# Group by player, look back exactly 7 days ('7D'), and count the matches
fatigue_df = match_players.groupby('player')['match_id'].rolling('7D').count().reset_index()
fatigue_df.rename(columns={'match_id': 'matches_last_7_days'}, inplace=True)

# Merge the fatigue score back with our original list
final_fatigue = pd.merge(match_players.reset_index(), fatigue_df, on=['player', 'date'])

# IMPORTANT FIX: The rolling window includes the current day's match. 
# We want the fatigue *entering* the match, so we subtract 1.
final_fatigue['fatigue_index'] = final_fatigue['matches_last_7_days'] - 1

# Drop the raw rolling column to keep it clean
final_fatigue.drop(columns=['matches_last_7_days'], inplace=True)

# 5. Save the Intelligence Data
os.makedirs('../data/processed', exist_ok=True)
final_fatigue.to_csv('../data/processed/player_fatigue.csv', index=False)

print("✅ Player Fatigue analysis complete! Data saved to 'data/processed/player_fatigue.csv'")
# Display a sample of high-fatigue scenarios (Players who played 2+ games in the last 7 days entering a match)
display(final_fatigue[final_fatigue['fatigue_index'] >= 2].head(10))

Starting Player Fatigue Analysis...
✅ Player Fatigue analysis complete! Data saved to 'data/processed/player_fatigue.csv'


,date,match_id,player,fatigue_index
2,2012-05-01,548348,A Ashish Reddy,2.0
3,2012-05-04,548352,A Ashish Reddy,2.0
4,2012-05-06,548356,A Ashish Reddy,2.0
5,2012-05-08,548359,A Ashish Reddy,2.0
6,2012-05-10,548329,A Ashish Reddy,3.0
11,2013-04-09,598048,A Ashish Reddy,2.0
12,2013-04-12,598010,A Ashish Reddy,2.0
13,2013-04-14,598013,A Ashish Reddy,2.0
14,2013-04-17,598018,A Ashish Reddy,2.0
15,2013-04-19,598021,A Ashish Reddy,2.0
